# Setup

## Import modules

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import torch
from datetime import datetime

# Import pipeline modules
from scoring import score_image
from selection import select
from metrics import MetricsTracker
from generation.UnconditionalGenerator import UnconditionalGenerator
from utils import save_image, set_seed
import shutil

# Config
OUTPUT_DIR = Path("outputs")

# Experiment parameters
NUM_PROMPTS = 30
NUM_TRIALS = 1
ROUNDS = 4        # r
BATCH_SIZE = 4    # B
SEED = 42

# Chosen strategies
SELECTION_STRATEGY = "argmax"

# Baseline scoring rubric (example)
# CLIP_RUBRIC = {"type": "clip", "text": "A photorealistic portrait of a dog", "weight": 1.0}
# BRIGHTNESS_RUBRIC = {"type": "brightness", "weight": 1.0}

set_seed(SEED)


## Load diffusion model

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    torch_dtype=torch.float16,
    # device_map="balanced",
)
pipe.enable_sequential_cpu_offload()

## Load dataset

In [ ]:
# Load dev and test datasets 
dev_dataset = pd.read_csv("data/dev_dataset.csv")
test_dataset = pd.read_csv("data/test_dataset.csv")

# Run simulation

In [ ]:
DATASET = dev_dataset[:NUM_PROMPTS]

metrics = MetricsTracker()
policy = UnconditionalGenerator(pipe)

 # based on date and time
RUN_DIR = OUTPUT_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = RUN_DIR / "images"
METRICS_CSV = RUN_DIR / "metrics.csv"

for trial in range(NUM_TRIALS):
    print("="*100)
    print(f"BEGINNING TRIAL {trial}")
    print("="*100)

    trial_id = f"trial{trial}"

    for item in tqdm(DATASET.to_dict(orient="records"), desc="prompts"):
        prompt_id = item["prompt_id"]
        prompt = item["prompt"]

        # Compile rubric
        rubric = {}
        for key, value in item.items():
            if key.startswith("rubric_"):
                rubric[key.replace("rubric_", "")] = value

        print(f"\nItem:")
        print(f"\tprompt_id={prompt_id}")
        print(f"\tprompt=\"{prompt}\"")
        print(f"\trubric={rubric}")
        print()

        prompt_outdir = IMAGES_DIR / prompt_id / trial_id

        for round in range(ROUNDS):
            first_round = (round == 0)

            print(f"\tRound {round}")
            round_outdir = prompt_outdir / f"round{round}"
            round_outdir.mkdir(parents=True, exist_ok=True)
            
            # Score and record metrics for each image in the batch
            user_scores = []

            # If not the first round, include the previous winner as the first image in the batch
            if not first_round:
                image_path = round_outdir / f"image{0}.png"
                previous_winner = metrics.clone_previous_winner(
                    prompt_id,
                    trial_id,
                    round, # the current round
                    image_path # the new image path
                )

                user_scores.append(previous_winner["user_score"])
                # clone the previous winner image to the new destination
                shutil.copy(previous_winner["image_path"], image_path)

            
            # Generate the necessary number of images using our method
            num_images = BATCH_SIZE - (0 if first_round else 1)
            new_images: list[Image.Image] = policy.generate(
                prompt,
                sampling_parameters={
                    "num_images_per_prompt": num_images,
                    "num_inference_steps": 1,
                    "guidance_scale": 7.5,
                    "width": 512,
                    "height": 512,
                    # "generator": torch.Generator(device="cuda").manual_seed(SEED), # this leads to different trials having the same images
                }
            )

            for image_idx, image in zip(range(0 if first_round else 1, BATCH_SIZE), new_images):
                image_path = round_outdir / f"image{image_idx}.png"
                
                user_score = score_image(image, rubric)
                user_scores.append(user_score)
                
                metrics.log(
                    prompt_id,
                    trial_id,
                    round,
                    image_idx,
                    image_path,
                    user_score,
                    image,
                    extra_info={
                        "prompt": prompt,
                    }
                )
                save_image(image, image_path)

            # Select favorite (index)
            chosen_image_idx = select(user_scores, strategy=SELECTION_STRATEGY)
            metrics.mark_chosen(
                prompt_id,
                trial_id,
                round,
                chosen_image_idx,
            )

            # Update policy
            policy.update(
                feedback={
                    # nothing for now
                }
            )

            print(f"\t\tImages saved to {round_outdir}")
            print(f"\t\tScores: {user_scores}")
            print(f"\t\tChosen index (using {SELECTION_STRATEGY}): {chosen_image_idx}")
            
            
        # Repeatedly save metrics per item, for safety
        metrics.save(str(METRICS_CSV)) # autoprints


In [ ]:
metrics.to_dataframe()

In [ ]:
# Import metrics from csv
metrics_df = pd.read_csv(METRICS_CSV)

In [ ]:
metrics_df